# Student Performance Analysis and Prediction System
**End-to-End Data Science & Machine Learning Portfolio Project**

This notebook walks through a complete Data Analytics and Machine Learning pipeline: Data Cleaning, Exploratory Data Analysis, Statistical Hypothesis Testing, Feature Engineering, Predictive ML Modeling, and Student Risk Classification. It also integrates an academic trajectory case study for student **BM EXCEL BLAZE**.

In [ ]:
# Core libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Scikit-Learn libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries successfully imported!")

## Phase 2: Data Loading and Inspection
We begin by loading the raw student dataset and inspecting its shape, columns, and missing values.

In [ ]:
raw_data_path = "../Dataset/student_data.csv"
if not os.path.exists(raw_data_path):
    # Fallback to local execution path
    raw_data_path = "Dataset/student_data.csv"
    
df_raw = pd.read_csv(raw_data_path)
print(f"Raw dataset shape: {df_raw.shape}")
print("\n--- First 5 Rows ---")
display(df_raw.head())
print("\n--- Missing Values Count ---")
print(df_raw.isnull().sum())

## Phase 3: Data Cleaning
Here we execute the data cleaning pipeline:
1. Remove duplicate entries.
2. Detect and handle outliers by capping values (Attendance at 100%, study hours capped, exam scores bounded to 0-100).
3. Impute missing values using robust statistics (median for numerical columns, mode for categorical parent education).
4. Verify Pass/Fail status integrity.

In [ ]:
# Import the data_preprocessing script functions
import sys
sys.path.append("..")
sys.path.append("Python_Scripts")
from data_preprocessing import clean_data

cleaned_df = clean_data(input_path=raw_data_path, output_path="../Dataset/student_data_cleaned.csv")
print(f"Cleaned dataset shape: {cleaned_df.shape}")
print(f"Remaining null values: {cleaned_df.isnull().sum().sum()}")

## Phase 4: Exploratory Data Analysis (EDA)
We analyze academic features to understand how attendance, study hours, gender, and parents' educational background affect student outcomes.

In [ ]:
colors = {'primary': '#2B4C7E', 'secondary': '#2A9D8F', 'accent': '#E76F51'}

# Final Exam Score Distribution
plt.figure(figsize=(8, 4))
sns.histplot(cleaned_df['Final_Exam_Score'], kde=True, color=colors['primary'], bins=20)
plt.axvline(60, color=colors['accent'], linestyle='--', label='Passing Mark (60)')
plt.title("Distribution of Final Exam Scores")
plt.xlabel("Score")
plt.ylabel("Count")
plt.legend()
plt.show()

# Bivariate: Study Hours vs Score
plt.figure(figsize=(8, 5))
sns.scatterplot(data=cleaned_df, x='Study_Hours', y='Final_Exam_Score', hue='Pass_Fail_Status', palette='viridis')
sns.regplot(data=cleaned_df, x='Study_Hours', y='Final_Exam_Score', scatter=False, color='red')
plt.title("Study Hours vs Final Exam Score")
plt.show()

## Phase 5: Statistical Analysis
We execute descriptive statistics and conduct hypothesis testing (T-Test, ANOVA, and Correlation metrics) to statistically validate our EDA findings.

In [ ]:
from statistical_analysis import run_statistical_analysis
run_statistical_analysis(input_path="../Dataset/student_data_cleaned.csv")

## Phase 7: Feature Engineering
We enrich our dataset by creating new meaningful features:
- **Attendance Category**: Mapping raw attendance percentage into brackets (Critical, Borderline, Good, Excellent).
- **Study Efficiency Index**: Daily study hours scaled by attendance level.
- **Risk Level**: Pre-exam vulnerability heuristic.

In [ ]:
from ml_modeling import engineer_features
df_eng = engineer_features(cleaned_df)
print("Engineered Columns Added:")
display(df_eng[['Attendance_Category', 'Study_Efficiency_Index', 'Risk_Level', 'Performance_Grade']].head())

## Phase 8: Machine Learning Model Training & Evaluation
We train Linear Regression, Decision Tree, Random Forest, and Gradient Boosting Regressors. Models are compared using MAE, RMSE, and $R^2$ scores.

In [ ]:
from ml_modeling import run_ml_pipeline
best_model_name, results = run_ml_pipeline(input_path="../Dataset/student_data_cleaned.csv")

## Phase 9: Student Risk Classification & Interventions
Using the trained model pipeline, we predict exam scores for all students, classify them into intervention priority levels (High, Medium, Low), and generate personalized intervention advice.

In [ ]:
from predict_risk import run_risk_prediction
risk_df = run_risk_prediction(data_path="../Dataset/student_data_engineered.csv", model_path="best_model.pkl")

## Case Study: Student BM EXCEL BLAZE
We load the transcript history for student **BM EXCEL BLAZE (Hallticket: 22D41A7210)**, analyze his SGPA trend from Sem 1 to 7, and perform linear trend regression to predict his Semester 8 performance.

In [ ]:
from blaze_analysis import analyze_blaze_records
analyze_blaze_records(input_path="../Dataset/blaze_academic_records.csv", output_dir="../Images")